# 1. Imports & Environment Setup

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

import evaluate
import librosa

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

C:\Users\edwin\OneDrive\Desktop\Convo AI\.gpuvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Load Dataset Splits

In [2]:
train_df = pd.read_csv("hdsd_train.csv")
val_df = pd.read_csv("hdsd_val.csv")
test_df = pd.read_csv("hdsd_test.csv")

(1633, 9)
(159, 9)
(210, 9)


## Inspect Dataset Sizes

In [46]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(1633, 9)
(159, 9)
(210, 9)


In [4]:
print(train_ds.column_names)
print(train_ds[0])

['audio', 'text', 'speaker', 'utterance', 'mic', 'n_words', 'duration_sec', 'text_norm', 'text_devnagari']
{'audio': 'C:\\Users\\edwin\\OneDrive\\Desktop\\Capstone\\hindi indic\\HDSD\\hindi_sent\\CF02\\CF02_S1_H01_M2.wav', 'text': 'aapakei hindii pasanda karanei para khushii huii', 'speaker': 'CF02', 'utterance': 'CF02_S1_H01', 'mic': 'M2', 'n_words': 7, 'duration_sec': 3.069875, 'text_norm': 'aapake hindi pasanda karane para khushi hui', 'text_devnagari': 'आपके हिन्दि पसन्द करने पर खुशि हुइ'}


# 3. Convert DataFrames to Hugging Face Datasets

In [3]:
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

# 4. Load Whisper Processor & Model

In [5]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="hi",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"Using device: {device}")

Using device: cuda


# 5. Baseline Inference Sanity Check

**Outcome:** The baseline Whisper model correctly transcribes a sample from the HDSD dataset. This sanity check verifies that audio loading, feature extraction, and inference are functioning correctly before fine-tuning begins.

In [47]:
sample = test_ds[0]

audio, sr = librosa.load(
    sample["audio"],
    sr=16000
)

inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

input_features = inputs.input_features.to(device)

with torch.no_grad():
    pred_ids = model.generate(input_features)

prediction = processor.batch_decode(
    pred_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

print("Reference:")
print(sample["text_devnagari"])

print()

print("Prediction:")
print(prediction)

Reference:
आपके हिन्दि पसन्द करने पर खुशि हुइ

Prediction:
 आपके हिंदी पसन्द कने पर खुषी हुई


### Preprocessing function

the preprocessing function is the bridge between our custom dataset and Whisper's expected input format.

Input {audio,text}

↓

Output {input_features, labels}

In [ ]:
def prepare_dataset(batch):

In [ ]:
train_ds = train_ds.map(...)
val_ds = val_ds.map(...)
test_ds = test_ds.map(...)

# 7. Data Collator

# 8. Evaluation Metrics

# 9. Training Configuration

# 10. Trainer Initialization

# 11. Fine-Tuning

# 12. Model Evaluation